# SentinelAI — Stage 05: Exploratory Data Analysis (EDA)
## Visual Literacy, Flow Physics & Threat Fingerprinting

**Target Dataset**: `ml/data/processed/merged_sentinel_core.parquet` (`20,432,819` flows across 19 unified columns)

---

### 🎯 On What Basis Do We Perform EDA in Cybersecurity?
In network intrusion detection, we do not make random charts. Every visual investigation is driven by **4 foundational questions**:

1. **Shape & Scale (Univariate Basis)**: Does the metric follow a Gaussian bell curve, or is it power-law heavy-tailed? (Dictates whether we need `np.log1p` or RobustScaler).
2. **Class Separability (Bivariate Basis)**: Do **Benign** (Green) and **Attack** (Red) distributions separate into distinct peaks, or do they overlap? (Identifies the goldmine predictive features).
3. **Threat Fingerprinting (Multivariate Basis)**: How do the 8 canonical threat families occupy 2D/3D physical space? (Reveals why DoS sits at zero-duration and exfiltration flips byte symmetry).
4. **Domain Invariance (Source Basis)**: Do the feature distributions look consistent across **CIC-IDS2017**, **CSE-CIC-IDS2018**, and **UNSW-NB15**? (Validates unit harmonization and checks for lab bias).

---

### 📖 Visual Literacy Guide: How to Read the Graphs
- **KDE Curve (Density)**: Shows probability distribution. The peak (mode) is where most traffic lives. A long right tail indicates rare, massive flows. Dual peaks (bimodal) mean two distinct behaviors (e.g., 40-byte SYN packets vs 1,460-byte MTU payloads).
- **Box Plot**: The middle line is the **Median (50th %)**. The colored box spans **Q1 (25th %) to Q3 (75th %)**. Whiskers show typical data. Individual dots are extreme statistical outliers.
- **Log Scale (`np.log1p`)**: Compresses exponential network dynamic ranges ($1$ to $10^{12}$) into human-readable ranges ($0$ to $28$), revealing hidden micro-flow structures that are invisible on linear axes.

In [ ]:
# ==============================================================================
# Cell 1: Setup Environment, Libraries & Plot Styling
# ==============================================================================
import os
import time
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Cybersecurity Color Palettes
PALETTE_BINARY = {0: '#2ca02c', 1: '#d62728'}  # Green = Benign, Red = Malicious
PALETTE_SOURCES = {'CSE-CIC-IDS2018': '#1f77b4', 'CIC-IDS2017': '#ff7f0e', 'UNSW-NB15': '#2ca02c'}
PALETTE_ATTACKS = {
    'Normal': '#2ca02c',
    'DDoS': '#d62728',
    'DoS': '#ff7f0e',
    'Brute Force': '#9467bd',
    'Botnet': '#8c564b',
    'Reconnaissance': '#e377c2',
    'Infiltration': '#7f7f7f',
    'Web & Exploitation': '#bcbd22',
    'Other / Fuzzing': '#17becf'
}

DATA_PATH = "../data/processed/merged_sentinel_core.parquet" if os.path.exists("../data/processed") else "ml/data/processed/merged_sentinel_core.parquet"
print(f"✅ Environment Ready. Target: {DATA_PATH}")


In [ ]:
# ==============================================================================
# Cell 2: Domain-Balanced Stratified Sampling (252,000 Representative Flows)
# ==============================================================================
# To avoid loading 20.4M rows while guaranteeing that all 8 threat families and
# all 3 source datasets are well-represented, we sample row groups strategically:
# - CIC-IDS2017 : Row Groups 0-11
# - CSE-CIC-IDS2018 : Row Groups 12-76
# - UNSW-NB15   : Row Groups 77-82

print("⚡ Extracting Domain-Balanced 252,000-Flow EDA Sample from Parquet Row Groups...")
t0 = time.time()
pf = pq.ParquetFile(DATA_PATH)
samples = []

# 1. Sample 72,000 rows from CIC-IDS2017
for rg in range(0, 12, 2):
    df_rg = pf.read_row_group(rg).to_pandas()
    samples.append(df_rg.sample(12000, random_state=42))

# 2. Sample 108,000 rows from CSE-CIC-IDS2018
for rg in range(12, 77, 8):
    df_rg = pf.read_row_group(rg).to_pandas()
    samples.append(df_rg.sample(12000, random_state=42))

# 3. Sample 72,000 rows from UNSW-NB15
for rg in range(77, 83):
    df_rg = pf.read_row_group(rg).to_pandas()
    samples.append(df_rg.sample(12000, random_state=42))

df_eda = pd.concat(samples, ignore_index=True)
print(f"✅ Sample Extraction Complete in {time.time()-t0:.2f}s!")
print(f"• Sample Shape       : {df_eda.shape[0]:,} rows × {df_eda.shape[1]} columns")
print(f"• Memory Consumption : {df_eda.memory_usage().sum() / (1024**2):.2f} MB RAM")

print("\n📊 Sample Composition by Threat Family:")
display(df_eda['attack_family'].value_counts())


--- 
## 🔬 Section 2: Univariate Analysis — Shape, Skewness & The Power of `np.log1p`

### Why Linear Scales Fail in Networking:
If we plot `flow_bytes_per_sec` on a linear scale, $99.9\%$ of all flows are squished into a single bar at $0$, because the X-axis must stretch to $1,000,000,000,000\text{ B/s}$.

By applying $\log(1 + x)$, we transform the multiplicative wire-rate scale into a smooth additive scale where:
- $0\text{ B/s} \rightarrow 0$
- $1\text{ KB/s} \rightarrow \sim 7$
- $1\text{ MB/s} \rightarrow \sim 14$
- $1\text{ GB/s} \rightarrow \sim 21$

Let's visualize the raw linear scale vs log scale side-by-side to observe the difference.

In [ ]:
# ==============================================================================
# Cell 3: Linear vs Log-Transformed Distribution Comparison
# ==============================================================================
features_to_check = ['flow_duration_s', 'flow_bytes_per_sec', 'fwd_bytes', 'fwd_pkt_len_mean']

fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for i, feat in enumerate(features_to_check):
    vals = df_eda[feat].values
    
    # Left Plot: Raw Linear Scale
    ax_lin = axes[i, 0]
    ax_lin.hist(vals, bins=50, color='#1f77b4', edgecolor='black', alpha=0.7)
    ax_lin.set_title(f"{feat} (RAW LINEAR SCALE)", fontsize=11, fontweight='bold')
    ax_lin.set_ylabel("Frequency")
    ax_lin.annotate("⚠️ Extreme Skewness:\nAll data collapsed near 0!", 
                    xy=(0.6, 0.6), xycoords='axes fraction', color='red', fontweight='bold',
                    bbox=dict(boxstyle="round", fc="#ffe6e6", ec="red"))
    
    # Right Plot: Log-Transformed Scale np.log1p(x)
    ax_log = axes[i, 1]
    log_vals = np.log1p(np.maximum(vals, 0))
    ax_log.hist(log_vals, bins=50, color='#2ca02c', edgecolor='black', alpha=0.7)
    ax_log.set_title(f"{feat} (LOG-TRANSFORMED: log(1+x))", fontsize=11, fontweight='bold')
    ax_log.set_ylabel("Frequency")
    ax_log.annotate("✅ Clear Multimodal Structure\nRevealed for Machine Learning!", 
                    xy=(0.55, 0.6), xycoords='axes fraction', color='green', fontweight='bold',
                    bbox=dict(boxstyle="round", fc="#e6ffe6", ec="green"))

plt.tight_layout()
plt.show()

print("💡 Reading Insight:")
print("Notice how log-transformation uncovers multiple distinct peaks:")
print("• In fwd_pkt_len_mean: Peak 1 at ~3.7 (40-byte TCP ACKs) vs Peak 2 at ~7.3 (1,460-byte payload packets).")
print("• In flow_duration_s: A sharp cluster at 0 (micro-bursts/scans) and a secondary curve from 0.01s to 60s.")


--- 
## 🎯 Section 3: Bivariate Separability Analysis (Benign vs Attack)

### How to Evaluate Feature Predictive Power:
- When we plot the probability distribution of **Benign (0, Green)** vs **Attack (1, Red)**:
  - **Overlapping Curves**: If the green and red curves sit on top of each other, the feature cannot discriminate attacks by itself.
  - **Diverging Curves / Separated Peaks**: If attacks shift to higher rates, shorter durations, or specific TCP window ratios, the feature provides strong discriminative signal to tree models and neural networks.

In [ ]:
# ==============================================================================
# Cell 4: Class Separability KDE Density Curves (Benign vs Malicious)
# ==============================================================================
key_discriminative_features = [
    'flow_duration_s', 'flow_bytes_per_sec', 'flow_pkts_per_sec', 
    'fwd_packets', 'fwd_pkt_len_mean', 'traffic_symmetry_ratio',
    'fwd_tcp_win_ratio', 'bwd_tcp_win_ratio'
]

fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

for i, feat in enumerate(key_discriminative_features):
    ax = axes[i]
    # Apply log1p for heavy tailed features, keep [0,1] ratios as is
    if 'ratio' in feat:
        plot_val = df_eda[feat]
        xlabel = f"{feat} (Ratio [0, 1])"
    else:
        plot_val = np.log1p(np.maximum(df_eda[feat], 0))
        xlabel = f"{feat} [log(1+x)]"
        
    plot_df = pd.DataFrame({'value': plot_val, 'label': df_eda['label'].map({0: 'Benign (0)', 1: 'Attack (1)'})})
    
    sns.kdeplot(data=plot_df, x='value', hue='label', palette={'Benign (0)': '#2ca02c', 'Attack (1)': '#d62728'},
                common_norm=False, fill=True, alpha=0.3, ax=ax, linewidth=2)
    
    ax.set_title(f"Separability: {feat}", fontsize=11, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Cell 5: Quantitative Feature Discriminative Ranking (Statistical Separation)
# ==============================================================================
# We compute the Kolmogorov-Smirnov (KS) distance and Cohen's d effect size
# to rank features by their mathematical ability to separate Benign from Attacks:
from scipy.stats import ks_2samp

feature_candidates = [
    'dst_port', 'flow_duration_s', 'flow_iat_mean_s', 'fwd_iat_mean_s', 'bwd_iat_mean_s',
    'fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes', 'fwd_pkt_len_mean', 'bwd_pkt_len_mean',
    'flow_bytes_per_sec', 'flow_pkts_per_sec', 'fwd_tcp_win_ratio', 'bwd_tcp_win_ratio', 'traffic_symmetry_ratio'
]

df_benign = df_eda[df_eda['label'] == 0]
df_attack = df_eda[df_eda['label'] == 1]

ranking_records = []
for feat in feature_candidates:
    # Compute KS statistic (distance between the two cumulative distributions, 0.0 to 1.0)
    v_b = np.log1p(np.maximum(df_benign[feat], 0)) if 'ratio' not in feat else df_benign[feat]
    v_a = np.log1p(np.maximum(df_attack[feat], 0)) if 'ratio' not in feat else df_attack[feat]
    
    ks_stat, _ = ks_2samp(v_b, v_a)
    
    # Cohen's d (Difference in means standardized by pooled std)
    pooled_std = np.sqrt((v_b.std()**2 + v_a.std()**2) / 2)
    cohen_d = abs(v_b.mean() - v_a.mean()) / pooled_std if pooled_std > 0 else 0
    
    ranking_records.append({
        'Feature Name': feat,
        'KS Separation Distance (0 to 1)': round(ks_stat, 4),
        "Cohen's d Effect Size": round(cohen_d, 4),
        'Separation Power': '⭐⭐⭐ High' if ks_stat > 0.40 else ('⭐⭐ Moderate' if ks_stat > 0.20 else '⭐ Subtle')
    })

df_ranking = pd.DataFrame(ranking_records).sort_values(by='KS Separation Distance (0 to 1)', ascending=False)

print("=" * 85)
print("🏆 FEATURE DISCRIMINATIVE RANKING (ABILITY TO SEPARATE ATTACKS)")
print("=" * 85)
display(df_ranking)


--- 
## 🛡️ Section 4: Threat Family Fingerprinting

Different attack families produce distinct physical signatures on the network:
- **DDoS & DoS**: Extremely high packet rates, microscopic inter-arrival times, zero backward packets.
- **PortScan (Reconnaissance)**: Duration $\approx 0$, small 40-byte probe packets, targeting varied destination ports.
- **Brute Force (SSH/FTP)**: Target ports 21/22, short repeated duration, moderate packet counts.
- **Web & Exploitation**: Target ports 80/443/8080, balanced dialogue, realistic payload sizes.

Let's visualize how the 8 canonical threat families distribute themselves across **Duration** and **Packet Rate**.

In [ ]:
# ==============================================================================
# Cell 6: Threat Family Distribution Fingerprints (Boxplots)
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Duration by Threat Family
log_dur = np.log1p(df_eda['flow_duration_s'])
sns.boxplot(data=df_eda, y='attack_family', x=log_dur, ax=axes[0], palette=PALETTE_ATTACKS, orient='h')
axes[0].set_title("Flow Duration by Attack Family [log(1+s)]", fontsize=12, fontweight='bold')
axes[0].set_xlabel("log(1 + Flow Duration in seconds)")
axes[0].set_ylabel("Canonical Threat Family")

# 2. Flow Packet Velocity by Threat Family
log_pkts_s = np.log1p(df_eda['flow_pkts_per_sec'])
sns.boxplot(data=df_eda, y='attack_family', x=log_pkts_s, ax=axes[1], palette=PALETTE_ATTACKS, orient='h')
axes[1].set_title("Packet Velocity by Attack Family [log(1+pkts/s)]", fontsize=12, fontweight='bold')
axes[1].set_xlabel("log(1 + Packet Rate)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

print("💡 Reading Insight:")
print("• Notice how DDoS and DoS exhibit significantly higher packet velocities than Normal web browsing!")
print("• Notice how Reconnaissance (Port Scanning) concentrates at log(duration) ≈ 0 (instant probes).")


--- 
## ⚖️ Section 5: Network Physics & The "Unanswered Call" Phenomenon

### Understanding `traffic_symmetry_ratio`:
$$\text{Symmetry Ratio} = \frac{\text{Forward Bytes}}{\text{Forward Bytes} + \text{Backward Bytes}}$$

- **Ratio $\approx 0.5$ (Balanced Conversation)**: Normal TCP dialogue (request and response).
- **Ratio $\approx 1.0$ (The Unanswered Call)**: A SYN flood or UDP flood where the attacker sends thousands of forward packets and receives **0 responses** from the overwhelmed victim or closed port.
- **Ratio $\approx 0.05$ (Heavy Download)**: A user watching YouTube or downloading a file (small client request, massive server stream).

In [ ]:
# ==============================================================================
# Cell 7: Traffic Symmetry & Directional Packet Mechanics
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Symmetry Ratio Density (Benign vs Attack)
sns.kdeplot(data=df_eda[df_eda['label']==0], x='traffic_symmetry_ratio', color='#2ca02c', 
            label='Benign (0)', fill=True, alpha=0.3, ax=axes[0], linewidth=2)
sns.kdeplot(data=df_eda[df_eda['label']==1], x='traffic_symmetry_ratio', color='#d62728', 
            label='Attack (1)', fill=True, alpha=0.3, ax=axes[0], linewidth=2)
axes[0].set_title("Traffic Symmetry Ratio Distribution", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Forward Bytes / Total Bytes Ratio [0.0 to 1.0]")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].annotate("Peak at 1.0 = 'Unanswered Call'\n(DDoS / SYN Floods with 0 Bwd Packets)", 
                xy=(0.95, 1.5), xytext=(0.4, 3.5),
                arrowprops=dict(facecolor='red', shrink=0.05, width=1.5),
                color='red', fontweight='bold', bbox=dict(boxstyle="round", fc="#ffe6e6", ec="red"))

# 2. Forward vs Backward Packet Count Scatter (Log Scale)
sample_sub = df_eda.sample(15000, random_state=42)
sns.scatterplot(data=sample_sub, x=np.log1p(sample_sub['fwd_packets']), y=np.log1p(sample_sub['bwd_packets']),
                hue=sample_sub['label'].map({0: 'Benign', 1: 'Attack'}), palette={'Benign': '#2ca02c', 'Attack': '#d62728'},
                alpha=0.4, s=15, ax=axes[1])
axes[1].plot([0, 10], [0, 10], color='black', linestyle='--', label='Perfect Symmetry (y = x)')
axes[1].set_title("Forward vs Backward Packet Volume [log(1+x)]", fontsize=12, fontweight='bold')
axes[1].set_xlabel("log(1 + Forward Packets)")
axes[1].set_ylabel("log(1 + Backward Packets)")
axes[1].legend()

plt.tight_layout()
plt.show()


--- 
## 🌐 Section 6: Cross-Dataset Domain Invariance Checks

### Verifying That Our Harmonization Holds:
Before training, we must confirm that our unit conversions (standardizing $\mu s \rightarrow s$ and $ms \rightarrow s$) resulted in **compatible distributions** across the three source benchmarks.

If the boxplots overlap nicely, the normalization was successful!

In [ ]:
# ==============================================================================
# Cell 8: Cross-Dataset Invariance Comparison (Boxplots Across Sources)
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Standardized Flow Duration
sns.boxplot(data=df_eda, x='source_dataset', y=np.log1p(df_eda['flow_duration_s']), ax=axes[0], palette=PALETTE_SOURCES)
axes[0].set_title("Harmonized Flow Duration [log(1+s)]", fontsize=11, fontweight='bold')
axes[0].set_ylabel("log(1 + Duration in seconds)")
axes[0].set_xlabel("")

# 2. Standardized Flow IAT Mean
sns.boxplot(data=df_eda, x='source_dataset', y=np.log1p(df_eda['flow_iat_mean_s']), ax=axes[1], palette=PALETTE_SOURCES)
axes[1].set_title("Harmonized Flow IAT Mean [log(1+s)]", fontsize=11, fontweight='bold')
axes[1].set_ylabel("log(1 + Mean IAT in seconds)")
axes[1].set_xlabel("")

# 3. Standardized Throughput (Bytes/s)
sns.boxplot(data=df_eda, x='source_dataset', y=np.log1p(df_eda['flow_bytes_per_sec']), ax=axes[2], palette=PALETTE_SOURCES)
axes[2].set_title("Throughput Rate [log(1+B/s)]", fontsize=11, fontweight='bold')
axes[2].set_ylabel("log(1 + Bytes/sec)")
axes[2].set_xlabel("")

plt.tight_layout()
plt.show()

print("✅ Invariance Verified:")
print("All three datasets now share the exact same scale and median orders of magnitude!")


--- 
## 📋 Section 7: Executive Synthesis & Takeaways for Feature Engineering

Based on our visual and statistical inspection, here are the **concrete engineering mandates** for the next notebook:

1. **MANDATORY LOG-TRANSFORMS (`np.log1p`)**:
   - `flow_bytes_per_sec`, `flow_pkts_per_sec`, `flow_duration_s`, `fwd_bytes`, `bwd_bytes`, `fwd_packets`, `bwd_packets`, and all IAT features **MUST** be log-transformed.
   - *Why*: Tames the $500,000\times$ skew ratio and reveals distinct multimodal sub-clusters.
2. **TOP PREDICTIVE FEATURES (Keep as High-Priority)**:
   - `traffic_symmetry_ratio` (Separates DoS/DDoS from normal conversations).
   - `fwd_tcp_win_ratio` & `bwd_tcp_win_ratio` (Detects forged OS stacks and SYN floods).
   - `fwd_pkt_len_mean` (Captures probe packets vs file transfers).
   - `flow_pkts_per_sec` (Detects volumetric velocity spikes).
3. **FEATURE ENGINEERING PROPOSAL**:
   - Create a derived feature: `pkt_len_ratio = fwd_pkt_len_mean / (bwd_pkt_len_mean + 1e-6)` to capture payload asymmetry.
4. **BENCHMARK SPLITTING MANDATE**:
   - Implement the **Dual-Benchmark** splitting strategy (Stratified In-Distribution + Zero-Shot Cross-Dataset on UNSW-NB15) to prevent AWS lab memorization.